# 🤖 DRGR VM — Google Colab

Этот ноутбук запускает полноценный AI-бэкенд (Ollama + drgr-bot VM) прямо в Google Colab
и подключает его к вашему локальному расширению через ngrok-туннель.

**Шаги:**
1. Запустите ячейку **«Установка»** (один раз)
2. Запустите **«Запуск Ollama + ngrok»**
3. Скопируйте ngrok URL и вставьте в расширение DRGR (📊 Visor VM → 🌐 Colab VM)

**Или:** режим поллинга — ноутбук сам опрашивает ваш сервер и обрабатывает задания.

In [ ]:
# ── 1. Установка зависимостей ────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd, **kw):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout[-2000:])
    if r.stderr and r.returncode != 0: print('[err]', r.stderr[-1000:])
    return r

# Ollama
run('curl -fsSL https://ollama.com/install.sh | sh')

# ngrok
run('pip install -q pyngrok')

# Python deps
run('pip install -q flask flask-cors requests')

print('\n✅ Установка завершена')

In [ ]:
# ── 2. Настройки ─────────────────────────────────────────────────────────────

# ngrok auth token (получить на https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = ''  # ← вставьте ваш токен
# ⚠ ВАЖНО: Не публикуйте и не коммитьте этот токен в репозиторий!

# Модели для автозагрузки (оставьте пустым список чтобы не качать)
PULL_MODELS = ['qwen2.5:1.5b']  # лёгкая модель для Colab Free

# URL вашего локального DRGR-сервера (для режима поллинга)
# Оставьте пустым если используете только ngrok-прокси
LOCAL_DRGR_URL = ''  # например: 'http://192.168.1.5:8080'

# Порт локального VM-сервера на Colab
VM_PORT = 8080

print('Настройки сохранены')

In [ ]:
# ── 3. Запуск Ollama ─────────────────────────────────────────────────────────
import subprocess, threading, time

def _ollama_server():
    subprocess.Popen(['ollama', 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

t = threading.Thread(target=_ollama_server, daemon=True)
t.start()
time.sleep(3)

import requests
try:
    r = requests.get('http://127.0.0.1:11434', timeout=5)
    print('✅ Ollama запущен:', r.text[:80])
except Exception as e:
    print('⚠ Ollama не отвечает:', e)

# Скачать модели
for model in PULL_MODELS:
    print(f'\n⬇ Скачиваю {model}…')
    r = subprocess.run(['ollama', 'pull', model],
                       capture_output=False, text=True)
    print('  статус:', r.returncode)

# Список моделей
try:
    models = requests.get('http://127.0.0.1:11434/api/tags', timeout=5).json()
    names = [m['name'] for m in models.get('models', [])]
    print('\n📋 Модели:', names)
except Exception as e:
    print('Ошибка получения моделей:', e)

In [ ]:
# ── 4. Мини-сервер DRGR на Colab ─────────────────────────────────────────────
# Запускает облегчённый Flask-сервер совместимый с drgr-bot VM API.
# Поддерживает: /health, /api/generate (proxy→Ollama), /v1/chat/completions

import flask, threading, json, requests as _req
from flask import Flask, request, jsonify
from flask_cors import CORS

colab_app = Flask('drgr_colab')
CORS(colab_app)

OLLAMA = 'http://127.0.0.1:11434'

@colab_app.route('/health')
def health():
    try:
        tags = _req.get(f'{OLLAMA}/api/tags', timeout=3).json()
        models = [m['name'] for m in tags.get('models', [])]
    except Exception:
        models = []
    return jsonify({
        'status': 'ok',
        'ollama': {'status': 'ok', 'models': models},
        'colab': True,
        'vm': {'status': 'ok'},
    })

@colab_app.route('/api/generate', methods=['POST'])
def api_generate():
    body = request.get_json(silent=True) or {}
    try:
        r = _req.post(f'{OLLAMA}/api/generate', json=body, timeout=120, stream=True)
        return flask.Response(r.iter_content(chunk_size=None),
                              content_type=r.headers.get('content-type','application/json'))
    except Exception as e:
        return jsonify({'error': str(e)}), 502

@colab_app.route('/api/chat', methods=['POST'])
def api_chat():
    body = request.get_json(silent=True) or {}
    try:
        r = _req.post(f'{OLLAMA}/api/chat', json=body, timeout=120, stream=True)
        return flask.Response(r.iter_content(chunk_size=None),
                              content_type=r.headers.get('content-type','application/json'))
    except Exception as e:
        return jsonify({'error': str(e)}), 502

@colab_app.route('/v1/chat/completions', methods=['POST'])
@colab_app.route('/api/v1/chat/completions', methods=['POST'])
def oai_chat():
    body = request.get_json(silent=True) or {}
    model = body.get('model', '')
    messages = body.get('messages', [])
    stream = body.get('stream', False)
    # Convert OAI format to Ollama
    prompt = '\n'.join(f"{m.get('role','user')}: {m.get('content','')}" for m in messages)
    payload = {'model': model, 'prompt': prompt, 'stream': stream}
    try:
        r = _req.post(f'{OLLAMA}/api/generate', json=payload, timeout=120, stream=stream)
        if stream:
            return flask.Response(r.iter_content(chunk_size=None),
                                  content_type='text/event-stream')
        data = r.json()
        return jsonify({
            'id': 'chatcmpl-colab',
            'object': 'chat.completion',
            'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': data.get('response','')}, 'finish_reason': 'stop'}],
            'model': model,
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 502

@colab_app.route('/v1/models', methods=['GET'])
@colab_app.route('/api/tags', methods=['GET'])
def list_models():
    try:
        return flask.Response(_req.get(f'{OLLAMA}/api/tags', timeout=5).content,
                              content_type='application/json')
    except Exception as e:
        return jsonify({'error': str(e)}), 502

# Start server in background thread
def _run_server():
    colab_app.run(host='0.0.0.0', port=VM_PORT, debug=False, use_reloader=False)

srv_thread = threading.Thread(target=_run_server, daemon=True)
srv_thread.start()
time.sleep(2)

try:
    h = requests.get(f'http://127.0.0.1:{VM_PORT}/health', timeout=5).json()
    print('✅ DRGR мини-сервер запущен, порт', VM_PORT)
    print('   Ollama модели:', h.get('ollama',{}).get('models',[]))
except Exception as e:
    print('⚠ Сервер не отвечает:', e)

In [ ]:
# ── 5. Создание ngrok туннеля ─────────────────────────────────────────────────
from pyngrok import ngrok, conf
import time

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()  # закрыть старые туннели
    tunnel = ngrok.connect(VM_PORT, 'http')
    NGROK_URL = tunnel.public_url
    # Prefer https
    if NGROK_URL.startswith('http://'):
        NGROK_URL = 'https://' + NGROK_URL[7:]

    print('\n' + '='*60)
    print('  ✅ DRGR Colab VM запущен!')
    print('='*60)
    print(f'  🌐 ngrok URL: {NGROK_URL}')
    print('='*60)
    print('\n  👆 Скопируйте URL выше и вставьте в расширение DRGR:')
    print('     Настройки → Remote VM URL   или')
    print('     Вкладка 🌐 Colab VM → поле URL → Сохранить')
    print()
    print(f'  Или запустите на ПК: ЗАПУСТИТЬ_COLAB_VM.bat {NGROK_URL}')

    # Авто-регистрация в локальном DRGR если указан URL
    if LOCAL_DRGR_URL:
        try:
            r = requests.post(f'{LOCAL_DRGR_URL}/colab/autostart',
                              json={'url': NGROK_URL}, timeout=10)
            d = r.json()
            if d.get('ok'):
                print(f'\n  ✅ Авто-зарегистрировано в локальном DRGR: {LOCAL_DRGR_URL}')
            else:
                print(f'  ⚠ Ответ DRGR: {d}')
        except Exception as e:
            print(f'  ℹ Авто-регистрация не удалась (это нормально, введите URL вручную): {e}')
else:
    NGROK_URL = ''
    print('⚠ NGROK_TOKEN не указан. Укажите токен в ячейке настроек.')
    print('  Получить бесплатно: https://dashboard.ngrok.com/get-started/your-authtoken')

In [ ]:
# ── 6. Режим поллинга (без ngrok) ────────────────────────────────────────────
# Используйте этот режим если ngrok недоступен.
# Ноутбук сам опрашивает ваш локальный DRGR-сервер и обрабатывает задания.
#
# Убедитесь что LOCAL_DRGR_URL задан в ячейке настроек!

import requests, time, threading

def _polling_worker(drgr_url: str, default_model: str = 'qwen2.5:1.5b',
                    interval: int = 3, stop_event=None):
    print(f'📬 Polling {drgr_url} каждые {interval}с…  (остановить: stop_polling.set())')
    while not (stop_event and stop_event.is_set()):
        try:
            r = requests.get(f'{drgr_url}/remote/jobs/pending', timeout=5)
            jobs = r.json().get('jobs', [])
            for job in jobs:
                jid     = job['id']
                payload = job.get('payload', {})
                prompt  = payload.get('prompt', '')
                model   = payload.get('model', default_model)
                print(f'  📥 Job {jid[:8]}: {prompt[:70]}')
                try:
                    resp = requests.post(
                        'http://127.0.0.1:11434/api/generate',
                        json={'model': model, 'prompt': prompt, 'stream': False},
                        timeout=120)
                    result = resp.json().get('response', resp.text[:500])
                except Exception as e:
                    result = f'[Ошибка Ollama {type(e).__name__}] {e}'
                requests.post(f'{drgr_url}/remote/jobs/{jid}/result',
                              json={'result': result}, timeout=10)
                print(f'  ✅ Job {jid[:8]} выполнен ({len(result)} символов)')
        except Exception as e:
            print(f'  ⚠ Polling error: {e}')
        time.sleep(interval)

if LOCAL_DRGR_URL:
    stop_polling = threading.Event()
    polling_thread = threading.Thread(
        target=_polling_worker,
        args=(LOCAL_DRGR_URL,),
        kwargs={'stop_event': stop_polling},
        daemon=True
    )
    polling_thread.start()
    print(f'✅ Polling запущен для {LOCAL_DRGR_URL}')
    print('  Чтобы остановить: stop_polling.set()')
else:
    print('ℹ LOCAL_DRGR_URL не задан — режим поллинга пропущен.')
    print('  Задайте LOCAL_DRGR_URL в ячейке настроек если хотите использовать поллинг.')

In [ ]:
# ── 7. Тест ──────────────────────────────────────────────────────────────────
import requests

# Тест Ollama
try:
    tags = requests.get('http://127.0.0.1:11434/api/tags', timeout=5).json()
    names = [m['name'] for m in tags.get('models', [])]
    print('✅ Ollama модели:', names)
except Exception as e:
    print('⚠ Ollama:', e)

# Тест DRGR мини-сервера
try:
    h = requests.get(f'http://127.0.0.1:{VM_PORT}/health', timeout=5).json()
    print('✅ DRGR сервер: статус =', h.get('status'))
except Exception as e:
    print('⚠ DRGR сервер:', e)

# Тест ngrok туннеля
if 'NGROK_URL' in dir() and NGROK_URL:
    try:
        h = requests.get(f'{NGROK_URL}/health', timeout=10).json()
        print(f'✅ ngrok туннель ({NGROK_URL}): статус =', h.get('status'))
    except Exception as e:
        print(f'⚠ ngrok туннель: {e}')
else:
    print('ℹ ngrok не настроен')

print('\n📋 Итог:')
print(f'  Ollama:  http://127.0.0.1:11434')
print(f'  Сервер:  http://127.0.0.1:{VM_PORT}')
if 'NGROK_URL' in dir() and NGROK_URL:
    print(f'  ngrok:   {NGROK_URL}  ← вставьте это в DRGR')

## 📝 Как использовать с блокнотом кода DRGR

После запуска ноутбука:

1. Скопируйте ngrok URL (например: `https://xxxx.ngrok-free.app`)
2. В расширении DRGR откройте вкладку **🌐 Colab VM**
3. Вставьте URL в поле и нажмите **💾 Сохранить URL**
4. В блокноте кода (Monaco) нажмите кнопку **☁ Remote** для генерации кода через Colab

Или запустите на ПК:
```
ЗАПУСТИТЬ_COLAB_VM.bat https://xxxx.ngrok-free.app
```

## 🔄 Режим поллинга (без ngrok)

Если нет ngrok-аккаунта — укажите `LOCAL_DRGR_URL` и используйте режим поллинга:
- В расширении: вкладка 🌐 Colab VM → кнопка **📬 Поллинг**
- Ноутбук будет автоматически забирать задания и возвращать результаты